# 08. Agent Evaluation — 9개 Evaluator로 보는 종합 에이전트 평가 랩

## 🎯 이 랩에서 무엇을 얻어가는가
이 랩이 끝나면 다음을 할 수 있게 됩니다.

1. Microsoft Foundry SDK가 제공하는 **9개 에이전트 evaluator**의 역할과 사용 시점을 구분할 수 있다.
2. 에이전트 평가를 두 축으로 본다.
   - **System Evaluation**: 결과(응답) 품질 — IntentResolution, TaskAdherence, Completeness 등
   - **Process Evaluation**: 과정(도구 호출) 품질 — ToolCallAccuracy
3. **OpenAI 메시지 스키마(`query[]`, `response[]`, `tool_definitions[]`)** 로 에이전트 trace를 표현하고 평가에 입력할 수 있다.
4. **`AIAgentConverter`** 로 Foundry Agent Service의 thread/run을 평가 입력으로 변환할 수 있다.
5. `evaluate()` 함수로 데이터셋 일괄 평가 + agent / 도메인 / 시나리오별 비교 분석을 할 수 있다.

> 입력 데이터: `data/agent_eval_traces.jsonl` (00 노트북에서 생성한 은행 도메인 trace)

---

## 📊 이 랩에서 다루는 평가 지표 (Evaluator) 종합 정리

### A. Agent 전용 (System / Process)
| Evaluator | 평가 질문 (한 줄) | 입력 | 점수 | 분류 |
|---|---|---|---|---|
| `IntentResolutionEvaluator` | 의도를 올바르게 파악했는가? | query, response (선택: tool_definitions) | 1~5 | System |
| `TaskAdherenceEvaluator` | 부여된 작업/역할을 준수했는가? | query, response | 1~5 | System |
| `ToolCallAccuracyEvaluator` | 올바른 도구를 올바른 인자로 호출했는가? | query, tool_definitions (선택: tool_calls, response) | 1~5 | Process |

### B. 일반 품질 (Quality)
| Evaluator | 평가 질문 (한 줄) | 입력 | 점수 |
|---|---|---|---|
| `ResponseCompletenessEvaluator` | 응답이 충분한 답을 했는가? | response, ground_truth | 1~5 |
| `CoherenceEvaluator` | 응답이 논리적으로 일관되는가? | query, response | 1~5 |
| `FluencyEvaluator` | 자연스럽게 읽히는가? | response | 1~5 |
| `RelevanceEvaluator` | 질문에 관련 있는가? | query, response | 1~5 |
| `GroundednessEvaluator` | 응답이 context에 근거하는가? | response, context | 1~5 |

### C. 안전성 (Safety, 이 랩의 확장 영역)
- `ContentSafetyEvaluator`, `IndirectAttackEvaluator`, `CodeVulnerabilityEvaluator` 등은 운영 배포 전에 함께 묶어 사용합니다 ([07_portal_sdk_evaluation.ipynb](07_portal_sdk_evaluation.ipynb) 참고).

### 🧪 실제 비즈니스 시나리오로 보면

#### ① System Evaluation: 사용자의 의도를 맞게 해결했는가
- **사용자 질문(query)**: `카드 분실 신고 어떻게 해?`
- **좋은 최종 응답(response)**: `앱 > 카드 > 분실신고 또는 1588-XXXX 콜센터에서 24시간 신고할 수 있고, 신고 즉시 카드 사용이 정지됩니다.`
- **나쁜 최종 응답(response)**: `카드 한도 변경은 모바일앱에서 신청할 수 있습니다.`
- **IntentResolution이 보는 지점**: 사용자의 목적이 `카드 분실 신고`였는지 정확히 파악했는가?
- **Relevance/Completeness가 보는 지점**: 답변이 질문에 직접 관련 있고, 필요한 채널·시간·즉시 정지 정보를 충분히 담았는가?
- 💼 **비즈니스 의미**: System Evaluation은 사용자가 최종적으로 받은 답변 품질을 봅니다. 상담사가 답을 잘못 말한 것과 같은 사용자 경험 문제입니다.

#### ② Process Evaluation: 맞는 도구를 맞는 인자로 호출했는가
- **사용자 질문(query)**: `ATM 출금 한도가 얼마야?`
- **사용 가능한 도구(tool_definitions)**: `search_policy(topic)`, `get_account_limit(account_id)`, `create_card_loss_report(card_id)`
- ✅ **좋은 tool call**: `search_policy(topic="ATM_WITHDRAWAL_LIMITS_2025")`
- ❌ **나쁜 tool call**: `create_card_loss_report(card_id="unknown")` 또는 `search_policy(topic="PARKING_LOT_RULES")`
- **ToolCallAccuracy가 보는 지점**: 답변 텍스트가 그럴듯한지보다, 에이전트가 **정확한 도구와 인자**를 골랐는가?
- 💼 **비즈니스 의미**: 에이전트가 우연히 맞는 답을 했더라도 잘못된 도구를 호출했다면 운영에서는 위험합니다. 특히 실제 계좌·카드 변경 도구는 process 평가가 필수입니다.

#### ③ 결과는 맞지만 과정이 위험한 경우
- **질문(query)**: `체크카드 해외결제 수수료 있어?`
- **도구 호출(tool call)**: `search_policy(topic="CARD_LIMIT_CHANGE_GUIDE")`
- **응답(response)**: `해외결제 시 국제브랜드 수수료 약 1%와 해외서비스 수수료가 부과됩니다.`
- **해석**: 응답이 우연히 맞아 보일 수 있어도, 호출한 문서는 카드 한도 변경 문서입니다.
- **봐야 할 evaluator**: `Relevance`나 `Completeness`만 보면 통과할 수 있지만, `ToolCallAccuracy`는 낮아야 합니다.
- 💼 **비즈니스 의미**: agent는 결과와 과정이 모두 맞아야 합니다. 결과만 평가하면 잘못된 내부 행동을 놓칩니다.

#### ④ 과정은 맞지만 최종 답변이 부족한 경우
- **질문(query)**: `대출 중도상환수수료는?`
- ✅ **좋은 tool call**: `search_policy(topic="MORTGAGE_PREPAYMENT_FEE_RULES")`
- ⚠️ **부족한 response**: `중도상환수수료는 1.2~1.5%입니다.`
- **해석**: 도구 선택은 맞았지만, 거치기간·면제 조건·일부상환 예외를 답하지 않았습니다.
- **봐야 할 evaluator**: `ToolCallAccuracy`는 높고, `ResponseCompleteness`는 낮을 수 있습니다.
- 💼 **비즈니스 의미**: 과정 평가와 결과 평가는 서로 대체 관계가 아닙니다. 둘을 같이 봐야 어느 레이어를 고칠지 압니다.

#### ⑤ `정보 부족` 상황에서의 agent 평가
- **질문(query)**: `마이너스 통장 개설 조건은?`
- **검색된 context**: `대환대출 안내 문서`
- ✅ **좋은 response**: `제공된 정보만으로는 마이너스 통장 개설 조건을 확인하기 어렵습니다.`
- ❌ **위험한 response**: `마이너스 통장은 연소득 3천만원 이상이면 누구나 개설할 수 있습니다.`
- **봐야 할 evaluator**: `Groundedness`, `TaskAdherence`, `IntentResolution`
- 💼 **비즈니스 의미**: agent 평가에서 중요한 것은 항상 정답을 말하는 것이 아니라, **근거가 없을 때 안전하게 멈추는 것**까지 포함합니다.

### Agent 전용 / 일반 품질 evaluator 차이
- **System evaluator**는 "**무엇을 답했는가**"를 평가합니다 → 응답 텍스트 위주.
- **Process evaluator**는 "**어떻게 답에 도달했는가**"를 평가합니다 → tool_calls / 도구 정의가 필수.
- 일반 품질 evaluator(B)는 단일 응답 텍스트 자체의 품질을 봅니다 → agent 외에도 재사용 가능.

### 비슷한 개념과 헷갈리지 않기
- `IntentResolution` ≠ `Relevance` — 후자는 텍스트 일치, 전자는 의도 충족.
- `ToolCallAccuracy` ≠ "도구가 성공했는가" — 도구 실행 결과의 성공/실패가 아니라, **에이전트가 올바른 도구를 적절한 인자로 골랐는지**입니다.
- Agent trace 입력은 **OpenAI messages 스키마**입니다 — 일반 QA evaluator에 그대로 넣을 수 없을 수 있습니다.

---

## 🧭 랩 구성 흐름

| 단계 | 내용 |
|---|---|
| 1 | 패키지 설치 / 데이터 로드 |
| 2 | 에이전트 평가 개요 + Evaluator 분류 |
| 3 | 에이전트 대화 데이터 구성 (OpenAI messages 스키마) |
| 4 | System Evaluation — IntentResolution / TaskAdherence / Quality 단건 평가 |
| 5 | Process Evaluation — ToolCallAccuracy (correct / wrong / hallucination / multi-step) |
| 6 | Foundry Agent Service 통합 — `AIAgentConverter` |
| 7 | 배치 평가 — `evaluate()` |
| 8 | 결과 분석 — 은행 도메인 하위 도메인(savings · loan · card · account) × 시나리오별 비교 |
| 9 | 요약 및 Best Practice |

> 💡 이 랩의 핵심 메시지: **에이전트는 "결과"와 "과정" 두 축으로 평가해야 한다. 결과만 봐서는 도구 선택이 잘못된 위험을, 과정만 봐서는 사용자 경험을 놓친다.**

---

# 08. 에이전트 평가 (Agent Evaluation) — 종합 가이드

이 노트북은 **Microsoft Foundry Agent Service의 에이전트 평가**를 종합적으로 다룹니다.

`azure-ai-evaluation` SDK가 제공하는 **9개 에이전트 전용 Evaluator**를 활용하여,
에이전트의 의도 파악, 작업 수행, 도구 호출 정확성을 체계적으로 평가합니다.

### 이 노트북에서 다루는 내용
1. 에이전트 평가 개요 및 Evaluator 분류
2. 에이전트 대화 데이터 구성 (OpenAI 메시지 스키마)
3. System Evaluation — 단일 평가
4. Process Evaluation — 도구 호출 평가
5. Foundry Agent Service 통합 (AIAgentConverter)
6. 배치 평가 (evaluate)
7. 결과 분석 — 도메인별 비교
8. 요약 및 Best Practice

### 참고 문서
- [Azure AI Foundry — Agent Evaluators](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/develop/agent-evaluate-sdk)
- [Agent Evaluation Overview](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/evaluation-approach-gen-ai)


## 1단계: 필수 패키지 설치

In [ ]:
%pip install --upgrade azure-ai-evaluation azure-ai-projects azure-identity python-dotenv

## 2단계: 에이전트 평가 개요

Azure AI Evaluation SDK는 에이전트를 **System Evaluation**과 **Process Evaluation** 두 관점에서 평가합니다.

### System Evaluation (End-to-End 결과 평가)
에이전트가 최종적으로 올바른 결과를 달성했는지 평가합니다.

| Evaluator | 설명 | 출력 | 비고 |
|-----------|------|------|------|
| **IntentResolutionEvaluator** | 사용자 의도를 올바르게 식별했는가? | 1-5 → Pass/Fail | preview |
| **TaskAdherenceEvaluator** | 시스템 지시사항을 준수했는가? | 1-5 → Pass/Fail | preview |
| **TaskCompletionEvaluator** | 작업을 완전히 완료했는가? | Pass/Fail | preview |
| **TaskNavigationEfficiencyEvaluator** | 최적 경로로 수행했는가? | Pass/Fail | ground_truth 필요 |

### Process Evaluation (단계별 프로세스 평가)
에이전트의 각 단계(도구 호출)가 올바른지 평가합니다.

| Evaluator | 설명 | 출력 | 비고 |
|-----------|------|------|------|
| **ToolCallAccuracyEvaluator** | 올바른 도구를 올바른 파라미터로 호출했는가? | 1-5 → Pass/Fail | 종합 평가 |
| **ToolSelectionEvaluator** | 불필요한 도구 없이 올바른 도구를 선택했는가? | Pass/Fail | |
| **ToolInputAccuracyEvaluator** | 도구에 올바른 파라미터를 전달했는가? | Pass/Fail | |
| **ToolOutputUtilizationEvaluator** | 도구 결과를 올바르게 활용했는가? | Pass/Fail | |
| **ToolCallSuccessEvaluator** | 도구 호출이 에러 없이 성공했는가? | Pass/Fail | |

### 에이전트 메시지 스키마 (OpenAI 형식)

에이전트 평가에서 사용하는 데이터는 OpenAI 대화 메시지 형식을 따릅니다:

```json
{
    "query": [
        {"role": "system", "content": "당신은 여행 예약 에이전트입니다."},
        {"role": "user", "content": "파리행 항공편을 예약해주세요"}
    ],
    "response": [
        {"role": "assistant", "content": [{"type": "tool_call", "name": "search_flights", "arguments": {"destination": "Paris"}}]},
        {"role": "tool", "content": [{"type": "tool_result", "tool_result": {"flight": "AF123"}}]},
        {"role": "assistant", "content": "파리행 AF123 항공편을 예약했습니다."}
    ]
}
```

### Tool Definitions 형식 (Azure AI Evaluation SDK 스키마, flat)

```json
[
  {
    "type": "function",
    "name": "search_flights",
    "description": "항공편을 검색합니다",
    "parameters": {
      "type": "object",
      "properties": {"destination": {"type": "string"}},
      "required": ["destination"]
    }
  }
]
```

- `query`: 시스템 프롬프트 + 사용자 메시지 (list of messages)
- `response`: 에이전트 응답 — tool_call, tool_result, 최종 텍스트 응답 (list of messages). 각 `tool_call`/`tool_result` 항목은 매칭 `tool_call_id` 필드를 가져야 함
- `tool_definitions`: 에이전트가 사용할 수 있는 도구 정의 (`type`/`name`/`description`/`parameters` 평탄화 스키마)

## 3단계: 에이전트 대화 데이터 구성

은행 도메인(savings · loan · card · account) 에이전트의 대화 데이터를 사용합니다 (00 노트북에서 생성).

### 데이터 시나리오

| 시나리오 | 설명 | 예상 평가 |
|----------|------|----------|
| ✅ 올바른 도구 + 올바른 파라미터 | 정상적인 에이전트 동작 | 높은 점수 |
| ✅ 다중 도구 호출 (multi-step) | 복잡한 질문에 여러 도구 순차 호출 | 높은 점수 |
| ❌ 잘못된 도구 선택 | 질문과 관련 없는 도구 호출 | 낮은 점수 |
| ❌ 잘못된 파라미터 | 올바른 도구지만 파라미터 오류 | 낮은 점수 |
| ❌ 환각 (도구 미사용) | 도구를 사용하지 않고 답변 생성 | 낮은 점수 |
| ❌ 의도 오해 | 사용자 질문과 다른 작업 수행 | 낮은 점수 |
| ❌ 도구 호출 실패 | 도구 호출 시 에러 발생 | 상황에 따라 |
| ❌ 불필요한 도구 호출 | 필요 없는 도구까지 호출 | 낮은 점수 |
| ❌ 작업 비준수 | 시스템 지시 무시 (off-topic 응답) | 낮은 점수 |


In [ ]:
import json
import os

# 데이터 파일 확인
data_path = "data/agent_eval_traces.jsonl"
assert os.path.exists(data_path), f"{data_path} 파일이 존재하지 않습니다."

# 데이터 로드 및 확인
traces = []
with open(data_path, "r", encoding="utf-8") as f:
    for line in f:
        traces.append(json.loads(line))

print(f"총 {len(traces)}개의 에이전트 대화 데이터 로드")
print(f"도메인 분포:")
domains = {}
for t in traces:
    d = t.get("domain", "unknown")
    domains[d] = domains.get(d, 0) + 1
for d, c in domains.items():
    print(f"  - {d}: {c}개")

print(f"\n시나리오 목록:")
for i, t in enumerate(traces):
    print(f"  [{i+1:2d}] {t['domain']:12s} | {t['scenario']}")

### 데이터 샘플 확인

첫 번째 데이터(정상 케이스)의 구조를 살펴봅니다.

In [ ]:
# 첫 번째 trace 상세 확인 (정상 케이스)
sample = traces[0]
print("=" * 60)
print(f"도메인: {sample['domain']} | 시나리오: {sample['scenario']}")
print("=" * 60)

print("\n📥 Query (입력):")
for msg in sample["query"]:
    print(f"  [{msg['role']}] {msg['content'][:80]}")

print("\n📤 Response (응답):")
for msg in sample["response"]:
    content = msg["content"]
    if isinstance(content, list):
        for item in content:
            if item["type"] == "tool_call":
                print(f"  [{msg['role']}] 🔧 tool_call: {item['name']}({json.dumps(item['arguments'], ensure_ascii=False)})")
            elif item["type"] == "tool_result":
                print(f"  [{msg['role']}] 📋 tool_result: {json.dumps(item['tool_result'], ensure_ascii=False)[:80]}")
    else:
        print(f"  [{msg['role']}] {content[:80]}")

print("\n🔧 Tool Definitions:")
for td in sample.get("tool_definitions", []):
    params = list(td["parameters"]["properties"].keys())
    print(f"  - {td['name']}({', '.join(params)}): {td['description']}")

## 4단계: System Evaluation — 단일 평가

System Evaluation은 에이전트가 **최종적으로 올바른 결과**를 달성했는지 평가합니다.

### 사용할 Evaluator
- **IntentResolutionEvaluator**: 사용자 의도를 올바르게 파악했는가? (1-5점)
- **TaskAdherenceEvaluator**: 시스템 지시사항을 준수했는가? (1-5점)
- **ToolCallAccuracyEvaluator**: 올바른 도구를 올바른 파라미터로 호출했는가? (1-5점)

> ⚠️ **주의**: 이 셀은 Azure OpenAI API 키가 필요합니다. `.env` 파일에 설정하세요.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

# 평가자(Evaluator)를 위한 모델 설정
model_config = {
    "azure_endpoint": os.getenv("AZURE_OPENAI_ENDPOINT"),
    "api_key": os.getenv("AZURE_OPENAI_API_KEY"),
    "azure_deployment": os.getenv("AZURE_OPENAI_DEPLOYMENT"),
}

print("Azure OpenAI 설정:")
print(f"  Endpoint: {model_config['azure_endpoint']}")
print(f"  Deployment: {model_config['azure_deployment']}")
print(f"  API Key: {'설정됨 ✅' if model_config['api_key'] else '미설정 ❌'}")

### 4-1. IntentResolutionEvaluator — 의도 파악 평가

에이전트가 사용자의 의도를 올바르게 식별하고 이해했는지 측정합니다.
- **입력**: `query` (대화 이력), `response` (에이전트 응답)
- **출력**: 1-5점 + Pass/Fail

In [ ]:
from azure.ai.evaluation import IntentResolutionEvaluator

intent_evaluator = IntentResolutionEvaluator(model_config=model_config)

# 좋은 예시: 올바른 의도 파악 (주택담보대출 금리 문의)
good_example = traces[0]
result_good = intent_evaluator(
    query=good_example["query"],
    response=good_example["response"],
)

print("✅ 좋은 예시 (올바른 의도 파악):")
print(f"  점수: {result_good.get('intent_resolution', 'N/A')}")
print(f"  결과: {result_good.get('intent_resolution_result', 'N/A')}")
print(f"  이유: {result_good.get('intent_resolution_reason', 'N/A')[:200]}")

In [ ]:
# 나쁜 예시: 의도 오해 (카드 분실 신고 → 카드 혜택 비교)
bad_example = traces[5]  # intent_misunderstanding
result_bad = intent_evaluator(
    query=bad_example["query"],
    response=bad_example["response"],
)

print("❌ 나쁜 예시 (의도 오해):")
print(f"  점수: {result_bad.get('intent_resolution', 'N/A')}")
print(f"  결과: {result_bad.get('intent_resolution_result', 'N/A')}")
print(f"  이유: {result_bad.get('intent_resolution_reason', 'N/A')[:200]}")

### 4-2. TaskAdherenceEvaluator — 작업 준수 평가

에이전트의 최종 응답이 시스템 프롬프트에 명시된 지시사항을 준수했는지 측정합니다.
- 역할을 벗어난 응답 감지
- 시스템 지시 무시 여부 확인

In [ ]:
from azure.ai.evaluation import TaskAdherenceEvaluator

task_evaluator = TaskAdherenceEvaluator(model_config=model_config)

# 좋은 예시: 시스템 지시 준수 (은행 상담 → 금리 안내)
good_example = traces[0]
result_good = task_evaluator(
    query=good_example["query"],
    response=good_example["response"],
)

print("✅ 좋은 예시 (작업 준수):")
print(f"  점수: {result_good.get('task_adherence', 'N/A')}")
print(f"  결과: {result_good.get('task_adherence_result', 'N/A')}")
print(f"  이유: {result_good.get('task_adherence_reason', 'N/A')[:200]}")

In [ ]:
# 나쁜 예시: 작업 비준수 (은행 상담인데 날씨 안내)
bad_example = traces[9]  # task_non_adherence
result_bad = task_evaluator(
    query=bad_example["query"],
    response=bad_example["response"],
)

print("❌ 나쁜 예시 (작업 비준수 — off-topic):")
print(f"  점수: {result_bad.get('task_adherence', 'N/A')}")
print(f"  결과: {result_bad.get('task_adherence_result', 'N/A')}")
print(f"  이유: {result_bad.get('task_adherence_reason', 'N/A')[:200]}")

## 5단계: Process Evaluation — 도구 호출 평가

Process Evaluation은 에이전트의 **각 도구 호출 단계**가 올바른지 평가합니다.

### ToolCallAccuracyEvaluator (종합)
올바른 도구를 올바른 파라미터로 호출했는지 종합 평가합니다.
- `tool_definitions` 필드가 필요합니다
- 도구 선택 + 파라미터 정확성을 함께 평가

> ⚠️ **주의**: 이 셀은 Azure OpenAI API 키가 필요합니다.

In [ ]:
from azure.ai.evaluation import ToolCallAccuracyEvaluator

tool_evaluator = ToolCallAccuracyEvaluator(model_config=model_config)

# 좋은 예시: 올바른 도구 호출
good_example = traces[0]  # correct_tool_correct_params
result_good = tool_evaluator(
    query=good_example["query"],
    response=good_example["response"],
    tool_definitions=good_example["tool_definitions"],
)

print("✅ 좋은 예시 (올바른 도구 호출):")
print(f"  점수: {result_good.get('tool_call_accuracy', 'N/A')}")
print(f"  결과: {result_good.get('tool_call_accuracy_result', 'N/A')}")
print(f"  이유: {result_good.get('tool_call_accuracy_reason', 'N/A')[:200]}")

In [ ]:
# 나쁜 예시: 잘못된 도구 선택 (다른 도메인의 도구를 호출하는 등)
bad_example = traces[12]  # wrong_tool selection
result_bad = tool_evaluator(
    query=bad_example["query"],
    response=bad_example["response"],
    tool_definitions=bad_example["tool_definitions"],
)

print("❌ 나쁜 예시 (잘못된 도구 선택):")
print(f"  점수: {result_bad.get('tool_call_accuracy', 'N/A')}")
print(f"  결과: {result_bad.get('tool_call_accuracy_result', 'N/A')}")
print(f"  이유: {result_bad.get('tool_call_accuracy_reason', 'N/A')[:200]}")


In [ ]:
# 나쁜 예시: 환각 (도구 미사용 — 금리를 임의로 답변)
hallucination = traces[4]  # hallucination_no_tool
result_halluc = tool_evaluator(
    query=hallucination["query"],
    response=hallucination["response"],
    tool_definitions=hallucination["tool_definitions"],
)

print("❌ 나쁜 예시 (환각 — 도구 미사용):")
print(f"  점수: {result_halluc.get('tool_call_accuracy', 'N/A')}")
print(f"  결과: {result_halluc.get('tool_call_accuracy_result', 'N/A')}")
print(f"  이유: {result_halluc.get('tool_call_accuracy_reason', 'N/A')[:200]}")

In [ ]:
# 다중 도구 호출 예시
multi_step = traces[1]  # multi_step_correct
result_multi = tool_evaluator(
    query=multi_step["query"],
    response=multi_step["response"],
    tool_definitions=multi_step["tool_definitions"],
)

print("✅ 다중 도구 호출 (자격 확인 → 금리 조회 → 상환액 계산):")
print(f"  점수: {result_multi.get('tool_call_accuracy', 'N/A')}")
print(f"  결과: {result_multi.get('tool_call_accuracy_result', 'N/A')}")
print(f"  이유: {result_multi.get('tool_call_accuracy_reason', 'N/A')[:200]}")

## 6단계: Foundry Agent Service 통합 (참고)

실제 **Azure AI Foundry Agent Service**를 사용하는 경우,
`AIAgentConverter`를 통해 에이전트의 대화 데이터를 평가 형식으로 자동 변환할 수 있습니다.

### AIAgentConverter 사용법

```python
from azure.ai.evaluation import AIAgentConverter
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

# Foundry 프로젝트 클라이언트 생성
project_client = AIProjectClient(
    credential=DefaultAzureCredential(),
    endpoint="https://<your-project>.services.ai.azure.com",
)

# 에이전트 대화 데이터를 평가 형식으로 변환
converter = AIAgentConverter(project_client)
converted_data = converter.convert(
    thread_id="thread_abc123",    # 에이전트 대화 스레드 ID
    run_id="run_xyz789",          # 에이전트 실행 ID
)

# 변환된 데이터는 query, response, tool_definitions 필드를 포함
print(converted_data.keys())  # dict_keys(['query', 'response', 'tool_definitions'])
```

### 변환된 데이터로 평가하기

```python
# 단일 항목 평가
result = intent_evaluator(
    query=converted_data["query"],
    response=converted_data["response"],
)

# JSONL 파일로 저장 후 배치 평가
import json
with open("data/foundry_agent_traces.jsonl", "w") as f:
    f.write(json.dumps(converted_data) + "\n")
```

> 💡 `AIAgentConverter`는 Foundry Agent Service의 thread/run에서 자동으로
> 메시지를 추출하고 OpenAI 메시지 스키마로 변환합니다.

In [ ]:
# [참고] Foundry Agent Service 통합 코드 (실행하려면 Foundry 프로젝트 필요)
# 아래 코드는 Foundry Agent Service를 사용하는 경우의 참고 코드입니다.

# from azure.ai.evaluation import AIAgentConverter
# from azure.ai.projects import AIProjectClient
# from azure.identity import DefaultAzureCredential
#
# project_client = AIProjectClient(
#     credential=DefaultAzureCredential(),
#     endpoint="https://<your-project>.services.ai.azure.com",
# )
#
# converter = AIAgentConverter(project_client)
# converted_data = converter.convert(
#     thread_id="thread_abc123",
#     run_id="run_xyz789",
# )
#
# # 변환된 데이터로 평가
# result = intent_evaluator(
#     query=converted_data["query"],
#     response=converted_data["response"],
# )
# print(result)

print("💡 Foundry Agent Service 통합은 실제 프로젝트 설정이 필요합니다.")
print("   위 주석 코드를 참고하여 연결하세요.")

## 7단계: 배치 평가 (evaluate)

`evaluate()` 함수를 사용하여 **전체 데이터셋에 대해 여러 Evaluator를 동시에 실행**합니다.

> ⚠️ **주의**: 이 셀은 Azure OpenAI API 키가 필요하며, 22개 데이터 × 3개 Evaluator = 66회 API 호출이 발생합니다.

In [ ]:
from azure.ai.evaluation import (
    evaluate,
    IntentResolutionEvaluator,
    TaskAdherenceEvaluator,
    ToolCallAccuracyEvaluator,
)

# 에이전트 전용 Evaluator 구성
agent_evaluators = {
    "intent_resolution": IntentResolutionEvaluator(model_config=model_config),
    "task_adherence": TaskAdherenceEvaluator(model_config=model_config),
    "tool_call_accuracy": ToolCallAccuracyEvaluator(model_config=model_config),
}

# 배치 평가 실행
result = evaluate(
    data="data/agent_eval_traces.jsonl",
    evaluators=agent_evaluators,
)

print("배치 평가 완료!")
print(f"\n📊 전체 메트릭 요약:")
for key, value in result.get("metrics", {}).items():
    print(f"  {key}: {value}")

In [ ]:
import pandas as pd

# 행별 결과를 DataFrame으로 변환
rows_df = pd.DataFrame(result["rows"])
print(f"결과 테이블 크기: {rows_df.shape}")
print(f"\n컬럼 목록:")
for col in rows_df.columns:
    print(f"  - {col}")

## 8단계: 결과 분석 — 도메인별 비교

선택한 시나리오(예: banking)의 도메인별, 평가 라벨별 결과를 분석합니다.

In [ ]:
# 도메인별 평균 점수 비교
# Azure AI Evaluation SDK 결과 컬럼: outputs.<evaluator_name>.<metric_name>
# 평가 점수는 보통 'outputs.<eval>.<eval>' 또는 gpt_<eval>; reason/result/threshold/token 등은 제외
_excluded_suffixes = ("_reason", "_result", "_details", "_threshold", "_prompt_tokens", "_completion_tokens", "_total_tokens", "_finish_reason", "_model", "_sample_input", "_sample_output")
score_cols = [c for c in rows_df.columns if c.startswith("outputs.") and not c.endswith(_excluded_suffixes)]

print("📊 도메인별 평균 점수:")
print("=" * 60)
domain_stats = rows_df.groupby("inputs.domain")[score_cols].mean(numeric_only=True)
print(domain_stats.to_string())

In [ ]:
# 시나리오별 상세 결과
result_cols = [c for c in rows_df.columns if c.endswith("_result")]

print("📋 시나리오별 Pass/Fail 결과:")
print("=" * 80)

display_cols = ["inputs.domain", "inputs.scenario_label"] + result_cols
available_cols = [c for c in display_cols if c in rows_df.columns]

if available_cols:
    scenario_results = rows_df[available_cols].copy()
    print(scenario_results.to_string(index=False))

In [ ]:
# Pass/Fail 비율 계산
print("📈 Evaluator별 Pass 비율:")
print("=" * 60)

for col in result_cols:
    if col in rows_df.columns:
        total = len(rows_df[col].dropna())
        if total > 0:
            pass_count = (rows_df[col] == "pass").sum()
            fail_count = (rows_df[col] == "fail").sum()
            evaluator_name = col.replace("outputs.", "").replace("_result", "")
            print(f"  {evaluator_name:30s} | Pass: {pass_count:3d} | Fail: {fail_count:3d} | Pass율: {pass_count/total*100:.1f}%")

In [ ]:
# 도메인별 Pass 비율 비교
print("📊 도메인별 Pass 비율 비교:")
print("=" * 70)

for domain in sorted(rows_df["inputs.domain"].dropna().unique()):
    domain_data = rows_df[rows_df["inputs.domain"] == domain]
    print(f"\n🏷️  {domain.upper()} (n={len(domain_data)}):")
    for col in result_cols:
        if col in domain_data.columns:
            total = len(domain_data[col].dropna())
            if total > 0:
                pass_count = (domain_data[col] == "pass").sum()
                evaluator_name = col.replace("outputs.", "").replace("_result", "")
                print(f"    {evaluator_name:30s} | Pass율: {pass_count/total*100:.1f}%")

## 9단계: 요약 및 Best Practice

### 📋 Evaluator 선택 가이드

| 평가 목적 | 권장 Evaluator | 설명 |
|-----------|---------------|------|
| 사용자 의도 파악 | `IntentResolutionEvaluator` | 에이전트가 질문을 이해했는지 |
| 시스템 지시 준수 | `TaskAdherenceEvaluator` | 역할/범위를 벗어나지 않는지 |
| 작업 완료 여부 | `TaskCompletionEvaluator` | 작업이 완전히 완료되었는지 |
| 최적 경로 | `TaskNavigationEfficiencyEvaluator` | ground_truth 대비 효율성 |
| 도구 호출 종합 | `ToolCallAccuracyEvaluator` | 도구 선택 + 파라미터 종합 |
| 도구 선택 | `ToolSelectionEvaluator` | 올바른 도구만 선택했는지 |
| 파라미터 정확성 | `ToolInputAccuracyEvaluator` | 파라미터가 정확한지 |
| 결과 활용 | `ToolOutputUtilizationEvaluator` | 도구 결과를 잘 활용했는지 |
| 호출 성공 | `ToolCallSuccessEvaluator` | 도구 호출 에러 없는지 |

### 🏗️ 프로덕션 적용 권장사항

1. **기본 평가 세트**: `IntentResolution` + `TaskAdherence` + `ToolCallAccuracy`
   - 에이전트의 핵심 동작을 종합적으로 평가

2. **세밀한 디버깅**: `ToolSelection` + `ToolInputAccuracy` + `ToolOutputUtilization`
   - 도구 호출의 어떤 단계에서 문제가 발생하는지 파악

3. **Foundry Agent Service 통합**: `AIAgentConverter` 사용
   - 실제 에이전트 대화를 자동으로 평가 형식으로 변환

4. **CI/CD 파이프라인 통합**: `evaluate()` 배치 평가 활용
   - Pass율 threshold 설정 → 자동 품질 게이트

5. **데이터 설계 원칙**:
   - 다양한 시나리오 포함 (정상, 오류, 엣지 케이스)
   - 도메인별 데이터 균형 유지
   - `tool_definitions` 항상 포함하여 도구 평가 활성화

### 🔗 참고 문서
- [Azure AI Foundry — Agent Evaluators](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/develop/agent-evaluate-sdk)
- [Evaluation Metrics](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/evaluation-metrics-built-in)
- [AIAgentConverter](https://learn.microsoft.com/en-us/python/api/azure-ai-evaluation/azure.ai.evaluation.aiagentconverter)